In [5]:
#了解ASII,unicode和utf-8编码
print('a')  #对应的ASCII码是97
print(ord('a'))  #ord函数可以查看字符对应的ASCII码
print(chr(97))  #chr函数可以查看ASCII码对应的字符
print('中')  #对应的Unicode编码是20013
print(ord('中'))  #ord函数可以查看字符对应的Unicode编码
print(chr(20013))  #chr函数可以查看Unicode编码对应的字符

#unicode只是解决了身份位置，但是没有解决怎么在内存里存储的问题
# 接下来就需要用到utf-8编码，前缀识别法
print('中'.encode('utf-8'))  #utf-8编码会把一个字符转换成多个字节存储
print(b'\xe4\xb8\xad'.decode('utf-8'))  #可以通过decode方法把字节转换成字符，这里是三个字节

a
97
a
中
20013
中
b'\xe4\xb8\xad'
中


In [9]:
# 以鲁为例子
print('鲁')
print(ord('鲁'))  #查看Unicode编码
print(bin(ord('鲁')))  #查看Unicode编码的二进制表示 0b1001110010000001这里为16bit 0bxxx
print(hex(ord('鲁')))  #查看Unicode编码的十六进制表示 0x9c81
print('鲁'.encode('utf-8'))  #查看utf-8编码 b'\xe9\xb2\x81'

鲁
40065
0b1001110010000001
0x9c81
b'\xe9\xb2\x81'


In [21]:
import regex

# 使用 [ ] 将空格包裹起来，防止被 VERBOSE 模式忽略
GPT2_PAT = regex.compile(r"""
    's|'t|'re|'ve|'m|'ll|'d       # 英语缩写
    | [ ]?\p{L}+                  # 字母序列（前面可能有空格 -> 用 [ ]? 表示）
    | [ ]?\p{N}+                  # 数字序列（前面可能有空格）
    | [ ]?[^\s\p{L}\p{N}]+        # 其他非字母数字字符序列（标点等）
    | \s+(?!\S)                   # 后面没有非空白字符的空白（通常是行尾空白）
    | \s+                         # 其他空白字符
""", regex.VERBOSE)

def gpt_pre_tokenize(text):
    tokens = GPT2_PAT.findall(text)
    return tokens

# 测试
text = "您好 人在哪"
tokens = gpt_pre_tokenize(text)
print(tokens)

['您好', ' 人在哪']


In [23]:
import collections

class BPE_From_Scratch:
    """
    步骤1，字节化和初始化，初始词表为0-255，每一个字节都是一个独立的token
    步骤2，构建统计字典，统计预分词后的token对出现的频率
    步骤3，寻找最高频率的字节对，合并为一个新的token，更新词表和文本
    步骤4，重复步骤2和3，直到达到预设的词表大小或者没有更多的token对可以合并
    性能优化：采用倒排索引来加速频率统计和更新文本
    终止条件：达到预设的词表大小或者没有更多的token对可以合并
    结果输出：最终的词表和编码后的文本
    """

    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        # 初始词表：0-255 的 ASCII/Byte 值
        # 格式: {token_id: bytes}
        self.vocab = {i: bytes([i]) for i in range(256)}
        # 记录合并规则：(token_a, token_b) -> new_token_id
        self.merges = {}
        self.next_token_id = 256

    def train(self, text):
        # -------------------------------------------------------------
        # 步骤1，字节化和初始化
        # -------------------------------------------------------------
        # 预分词 GPT2_PAT 正则使用
        words = gpt_pre_tokenize(text)
        
        # 将文本转换为 token 列表的频率字典
        # 例如: "hello" -> (104, 101, 108, 108, 111)
        word_freqs = collections.defaultdict(int)
        for word in words:
            # 将单词转为 UTF-8 字节流，再转为整数元组
            word_bytes = tuple(word.encode("utf-8"))
            word_freqs[word_bytes] += 1
        
        print(f"[Info] 初始词表大小: {len(self.vocab)}")
        print(f"[Info] 唯一单词数量: {len(word_freqs)}")

        # -------------------------------------------------------------
        # 步骤4，重复步骤2和3
        # -------------------------------------------------------------
        while len(self.vocab) < self.vocab_size:
            # -------------------------------------------------------------
            # 步骤2，构建统计字典
            # -------------------------------------------------------------
            pairs = collections.defaultdict(int)
            for word_tuple, freq in word_freqs.items():
                # 遍历当前单词中的所有相邻对
                for i in range(len(word_tuple) - 1):
                    pair = (word_tuple[i], word_tuple[i+1])
                    pairs[pair] += freq  # 加权统计（乘以单词出现的频率）

            # -------------------------------------------------------------
            # 终止条件检查
            # -------------------------------------------------------------
            if not pairs:
                print("[Stop] 没有更多的 token 对可以合并")
                break

            # -------------------------------------------------------------
            # 步骤3，寻找最高频率的字节对，合并
            # -------------------------------------------------------------
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]

            # 更新词表
            new_token_id = self.next_token_id
            self.vocab[new_token_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_token_id
            self.next_token_id += 1

            # 更新文本（这里的“文本”是 word_freqs 字典）
            # 性能优化：只更新包含 best_pair 的单词，而不是全量扫描
            new_word_freqs = collections.defaultdict(int)
            for word_tuple, freq in word_freqs.items():
                # 如果这个单词里不包含我们要合并的对，直接保留
                if best_pair[0] not in word_tuple: 
                    # 简单的过滤优化，实际可以使用更复杂的倒排索引
                    new_word_freqs[word_tuple] = freq
                    continue
                
                # 执行合并操作
                new_tuple = []
                i = 0
                while i < len(word_tuple):
                    # 检查是否匹配 best_pair
                    if i < len(word_tuple) - 1 and word_tuple[i] == best_pair[0] and word_tuple[i+1] == best_pair[1]:
                        new_tuple.append(new_token_id)
                        i += 2 # 跳过两个
                    else:
                        new_tuple.append(word_tuple[i])
                        i += 1
                new_word_freqs[tuple(new_tuple)] = freq
            
            word_freqs = new_word_freqs

            # 打印日志
            if len(self.vocab) % 5 == 0: # 每合并5次打印一次
                print(f"Merge: {best_pair} -> {new_token_id} (Count: {best_count})")

        # -------------------------------------------------------------
        # 结果输出
        # -------------------------------------------------------------
        print(f"\n[Done] 训练完成。最终词表大小: {len(self.vocab)}")
        return self.vocab, self.merges

    def encode(self, text):
        """
        使用训练好的 merges 规则对新文本进行编码
        """
        words = text.split()
        encoded_ids = []
        
        for word in words:
            # 初始状态：转为字节 ID
            word_ids = list(word.encode("utf-8"))
            
            while len(word_ids) >= 2:
                # 寻找当前单词中，所有可能的 pair，找到在 merges 中优先级最高（最早加入）的那个
                # 注意：实际 BPE 实现中，应该按照 merges 的加入顺序来合并
                stats = {}
                for i in range(len(word_ids)-1):
                    pair = (word_ids[i], word_ids[i+1])
                    if pair in self.merges:
                        # 我们需要知道这个 pair 是第几个被 merge 的，这里简化逻辑
                        stats[pair] = pair # 简单占位，实际应存 priority
                
                if not stats:
                    break
                
                # 这里为了简单演示，假设只要在 merges 里就合并（实际应选最早 merge 的）
                # 找到当前单词中能合并的一对（这里简单取第一个匹配的）
                pair_to_merge = list(stats.keys())[0] 
                new_id = self.merges[pair_to_merge]
                
                # 执行替换
                new_ids = []
                i = 0
                while i < len(word_ids):
                    if i < len(word_ids) - 1 and word_ids[i] == pair_to_merge[0] and word_ids[i+1] == pair_to_merge[1]:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(word_ids[i])
                        i += 1
                word_ids = new_ids
            
            encoded_ids.extend(word_ids)
            
        return encoded_ids

# -------------------------------------------------------------
# 测试代码
# -------------------------------------------------------------
if __name__ == "__main__":
    # 模拟一段重复度高的数据，方便观察合并效果
    # "aa" -> 97, 97 -> 合并
    text_data = "aa ab aa ab ac aa ad aa ab" 
    
    # 实例化 BPE
    bpe = BPE_From_Scratch(vocab_size=260) # 初始256，我们只想多训练4个词
    
    # 训练
    vocab, merges = bpe.train(text_data)
    
    # 打印部分结果
    print("\n--- Merges (合并规则) ---")
    for pair, new_id in merges.items():
        print(f"Pair {pair} -> New ID {new_id}")
        
    # 测试编码
    test_str = "ab aa ac"
    encoded = bpe.encode(test_str)
    print(f"\n--- Encode Test ---")
    print(f"原文: {test_str}")
    print(f"编码 ID: {encoded}")

[Info] 初始词表大小: 256
[Info] 唯一单词数量: 5
Merge: (97, 97) -> 259 (Count: 1)

[Done] 训练完成。最终词表大小: 260

--- Merges (合并规则) ---
Pair (32, 97) -> New ID 256
Pair (256, 98) -> New ID 257
Pair (256, 97) -> New ID 258
Pair (97, 97) -> New ID 259

--- Encode Test ---
原文: ab aa ac
编码 ID: [97, 98, 259, 97, 99]
